# Quantization-Sparsity Aware Training with NNCF, using PyTorch framework

This notebook is based on [ImageNet training in PyTorch](https://github.com/pytorch/examples/blob/master/imagenet/main.py).

The goal of this notebook is to demonstrate how to use the Neural Network Compression Framework [NNCF](https://github.com/openvinotoolkit/nncf) to optimize a PyTorch model for inference with OpenVINO Toolkit. The optimization process contains the following steps:

* Pruning the model using magnitude-based unstructured sparsity via `nncf.prune()`
* Quantizing the pruned model to `INT8` using `nncf.quantize()`
* Fine-tuning the quantized model to recover accuracy
* Exporting optimized and original models to OpenVINO IR
* Measuring and comparing the performance of models.

For more advanced usage, refer to these [examples](https://github.com/openvinotoolkit/nncf/tree/develop/examples).

This tutorial uses the ResNet-18 model with the Tiny ImageNet-200 dataset.




<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/pytorch-quantization-sparsity-aware-training/pytorch-quantization-sparsity-aware-training.ipynb" />


#### Table of contents:

- [Imports and Settings](#Imports-and-Settings)
- [Pre-train Floating-Point Model](#Pre-train-Floating-Point-Model)
    - [Train Function](#Train-Function)
    - [Validate Function](#Validate-Function)
    - [Helpers](#Helpers)
    - [Get a Pre-trained FP32 Model](#Get-a-Pre-trained-FP32-Model)
- [Apply Pruning (Sparsity)](#Apply-Pruning-(Sparsity))
- [Fine-tune the Pruned Model](#Fine-tune-the-Pruned-Model)
- [Quantize the Pruned Model](#Quantize-the-Pruned-Model)
- [Fine-tune the Quantized Model (QAT)](#Fine-tune-the-Quantized-Model-(QAT))
- [Export INT8 Sparse Model to OpenVINO IR](#Export-INT8-Sparse-Model-to-OpenVINO-IR)
- [Benchmark Model Performance by Computing Inference Time](#Benchmark-Model-Performance-by-Computing-Inference-Time)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

In [1]:
%pip install -q --extra-index-url https://download.pytorch.org/whl/cpu  "openvino>=2024.0.0" "torch" "torchvision" "tqdm" "nncf>=3.0.0" "numpy<2; sys_platform == 'darwin'"

Note: you may need to restart the kernel to use updated packages.


## Imports and Settings
[back to top ⬆️](#Table-of-contents:)

On Windows, add the required C++ directories to the system PATH.

Import NNCF and all auxiliary packages from your Python code.
Set a name for the model, and the image width and height that will be used for the network. Also define paths where PyTorch and OpenVINO IR versions of the models will be stored. 

> **NOTE**: All NNCF logging messages below ERROR level (INFO and WARNING) are disabled to simplify the tutorial. For production use, it is recommended to enable logging by removing ```set_log_level(logging.ERROR)```.

In [2]:
import time
import warnings  # To disable warnings on export model
from pathlib import Path

import torch

import torch.nn as nn
import torch.nn.parallel
import torch.optim
import torch.utils.data
import torch.utils.data.distributed
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torchvision.models as models

import openvino as ov
from torch.jit import TracerWarning

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")

MODEL_DIR = Path("model")
OUTPUT_DIR = Path("output")
# DATA_DIR = Path("...")  # Insert path to folder containing imagenet folder
# DATASET_DIR = DATA_DIR / "imagenet"

Using cpu device


In [3]:
# Fetch `notebook_utils` module
import zipfile
import requests

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)
from notebook_utils import download_file, device_widget

DATA_DIR = Path("data")


def download_tiny_imagenet_200(
    data_dir: Path,
    url="http://cs231n.stanford.edu/tiny-imagenet-200.zip",
    tarname="tiny-imagenet-200.zip",
):
    archive_path = data_dir / tarname
    download_file(url, directory=data_dir, filename=tarname)
    zip_ref = zipfile.ZipFile(archive_path, "r")
    zip_ref.extractall(path=data_dir)
    zip_ref.close()


def prepare_tiny_imagenet_200(dataset_dir: Path):
    # Format validation set the same way as train set is formatted.
    val_data_dir = dataset_dir / "val"
    val_annotations_file = val_data_dir / "val_annotations.txt"
    with open(val_annotations_file, "r") as f:
        val_annotation_data = map(lambda line: line.split("\t")[:2], f.readlines())
    val_images_dir = val_data_dir / "images"
    for image_filename, image_label in val_annotation_data:
        from_image_filepath = val_images_dir / image_filename
        to_image_dir = val_data_dir / image_label
        if not to_image_dir.exists():
            to_image_dir.mkdir()
        to_image_filepath = to_image_dir / image_filename
        from_image_filepath.rename(to_image_filepath)
    val_annotations_file.unlink()
    val_images_dir.rmdir()


DATASET_DIR = DATA_DIR / "tiny-imagenet-200"
if not DATASET_DIR.exists():
    download_tiny_imagenet_200(DATA_DIR)
    prepare_tiny_imagenet_200(DATASET_DIR)
    print(f"Successfully downloaded and prepared dataset at: {DATASET_DIR}")

BASE_MODEL_NAME = "resnet18"
image_size = 64

OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

# Paths where PyTorch and OpenVINO IR models will be stored.
fp32_pth_path = Path(MODEL_DIR / (BASE_MODEL_NAME + "_fp32")).with_suffix(".pth")
fp32_ir_path = fp32_pth_path.with_suffix(".xml")
int8_sparse_ir_path = Path(MODEL_DIR / (BASE_MODEL_NAME + "_int8_sparse")).with_suffix(".xml")

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("pytorch-quantization-sparsity-aware-training.ipynb")

tiny-imagenet-200.zip:   0%|          | 0.00/237M [00:00<?, ?B/s]

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



Successfully downloaded and prepared dataset at: data/tiny-imagenet-200


### Train Function
[back to top ⬆️](#Table-of-contents:)


In [4]:
def train(train_loader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter("Time", ":3.3f")
    losses = AverageMeter("Loss", ":2.3f")
    top1 = AverageMeter("Acc@1", ":2.2f")
    top5 = AverageMeter("Acc@5", ":2.2f")
    progress = ProgressMeter(
        len(train_loader),
        [batch_time, losses, top1, top5],
        prefix="Epoch:[{}]".format(epoch),
    )

    # Switch to train mode.
    model.train()

    end = time.time()
    for i, (images, target) in enumerate(train_loader):
        images = images.to(device)
        target = target.to(device)

        # Compute output.
        output = model(images)
        loss = criterion(output, target)

        # Measure accuracy and record loss.
        acc1, acc5 = accuracy(output, target, topk=(1, 5))
        losses.update(loss.item(), images.size(0))
        top1.update(acc1[0], images.size(0))
        top5.update(acc5[0], images.size(0))

        # Compute gradient and do opt step.
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Measure elapsed time.
        batch_time.update(time.time() - end)
        end = time.time()

        print_frequency = 50
        if i % print_frequency == 0:
            progress.display(i)

### Validate Function
[back to top ⬆️](#Table-of-contents:)


In [5]:
def validate(val_loader, model, criterion):
    batch_time = AverageMeter("Time", ":3.3f")
    losses = AverageMeter("Loss", ":2.3f")
    top1 = AverageMeter("Acc@1", ":2.2f")
    top5 = AverageMeter("Acc@5", ":2.2f")
    progress = ProgressMeter(len(val_loader), [batch_time, losses, top1, top5], prefix="Test: ")

    # Switch to evaluate mode.
    model.eval()

    with torch.no_grad():
        end = time.time()
        for i, (images, target) in enumerate(val_loader):
            images = images.to(device)
            target = target.to(device)

            # Compute output.
            output = model(images)
            loss = criterion(output, target)

            # Measure accuracy and record loss.
            acc1, acc5 = accuracy(output, target, topk=(1, 5))
            losses.update(loss.item(), images.size(0))
            top1.update(acc1[0], images.size(0))
            top5.update(acc5[0], images.size(0))

            # Measure elapsed time.
            batch_time.update(time.time() - end)
            end = time.time()

            print_frequency = 10
            if i % print_frequency == 0:
                progress.display(i)

        print(" * Acc@1 {top1.avg:.3f} Acc@5 {top5.avg:.3f}".format(top1=top1, top5=top5))
    return top1.avg

### Helpers
[back to top ⬆️](#Table-of-contents:)


In [6]:
class AverageMeter(object):
    """Computes and stores the average and current value"""

    def __init__(self, name, fmt=":f"):
        self.name = name
        self.fmt = fmt
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

    def __str__(self):
        fmtstr = "{name} {val" + self.fmt + "} ({avg" + self.fmt + "})"
        return fmtstr.format(**self.__dict__)


class ProgressMeter(object):
    def __init__(self, num_batches, meters, prefix=""):
        self.batch_fmtstr = self._get_batch_fmtstr(num_batches)
        self.meters = meters
        self.prefix = prefix

    def display(self, batch):
        entries = [self.prefix + self.batch_fmtstr.format(batch)]
        entries += [str(meter) for meter in self.meters]
        print("\t".join(entries))

    def _get_batch_fmtstr(self, num_batches):
        num_digits = len(str(num_batches // 1))
        fmt = "{:" + str(num_digits) + "d}"
        return "[" + fmt + "/" + fmt.format(num_batches) + "]"


def accuracy(output, target, topk=(1,)):
    """Computes the accuracy over the k top predictions for the specified values of k"""
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)

        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))

        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
            res.append(correct_k.mul_(100.0 / batch_size))
        return res

### Get a Pre-trained FP32 Model
[back to top ⬆️](#Table-of-contents:)

А pre-trained floating-point model is a prerequisite for quantization. It can be obtained by tuning from scratch with the code below. 

In [7]:
num_classes = 1000
init_lr = 1e-4
batch_size = 128
epochs = 20

# model = models.resnet50(pretrained=True)
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(in_features=512, out_features=200, bias=True)
model.to(device)


# Data loading code.
train_dir = DATASET_DIR / "train"
val_dir = DATASET_DIR / "val"
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_dataset = datasets.ImageFolder(
    train_dir,
    transforms.Compose(
        [
            transforms.Resize([image_size, image_size]),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            normalize,
        ]
    ),
)
val_dataset = datasets.ImageFolder(
    val_dir,
    transforms.Compose(
        [
            transforms.Resize([256, 256]),
            transforms.CenterCrop([image_size, image_size]),
            transforms.ToTensor(),
            normalize,
        ]
    ),
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=1,
    pin_memory=True,
    sampler=None,
)

val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=1, pin_memory=True)

# Define loss function (criterion) and optimizer.
criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=init_lr)

/home/maleksandr/test_notebooks/nncf-tests/openvino_notebooks/venv12/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/maleksandr/test_notebooks/nncf-tests/openvino_notebooks/venv12/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/maleksandr/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████████████████████████████████████████████████████████████████████████| 44.7M/44.7M [00:01<00:00, 41.9MB/s]


Export the `FP32` model to OpenVINO™ Intermediate Representation, to benchmark it in comparison with the `INT8` model.

In [8]:
dummy_input = torch.randn(1, 3, image_size, image_size).to(device)

ov_model = ov.convert_model(model, example_input=dummy_input, input=[1, 3, image_size, image_size])
ov.save_model(ov_model, fp32_ir_path, compress_to_fp16=False)
print(f"FP32 model was exported to {fp32_ir_path}.")

FP32 model was exported to model/resnet18_fp32.xml.


## Apply Pruning (Sparsity)

[back to top ⬆️](#Table-of-contents:)

First, we apply magnitude-based unstructured pruning using `nncf.prune()`. This zeros out weights with small magnitudes, making the model sparse. We use a moderate sparsity ratio of 30% and apply Batch Norm adaptation to stabilize the pruned model before fine-tuning.

In [ ]:
import nncf

example_input = torch.randn(1, 3, image_size, image_size).to(device)

# Apply magnitude-based unstructured pruning with 30% sparsity ratio
pruned_model = nncf.prune(
    model,
    mode=nncf.PruneMode.UNSTRUCTURED_MAGNITUDE_GLOBAL,
    ratio=0.3,
    examples_inputs=example_input,
)

# Print pruning statistics
print(nncf.pruning_statistic(pruned_model))

# Batch Norm adaptation for better accuracy after pruning
def bn_transform_fn(batch):
    inputs, _ = batch
    return inputs.to(device)

bn_dataset = nncf.Dataset(train_loader, bn_transform_fn)
pruned_model = nncf.batch_norm_adaptation(
    pruned_model,
    calibration_dataset=bn_dataset,
    num_iterations=200,
)
print("Batch Norm adaptation completed.")

┍━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┯━━━━━━━━━━━━━━━━━━┯━━━━━━━━━━━━━━━━━┑
│ Parameter's name             │ Shape            │   Pruning ratio │
┝━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┿━━━━━━━━━━━━━━━━━━┿━━━━━━━━━━━━━━━━━┥
│ conv1.weight                 │ (64, 3, 7, 7)    │           0.202 │
├──────────────────────────────┼──────────────────┼─────────────────┤
│ fc.weight                    │ (200, 512)       │           0.095 │
├──────────────────────────────┼──────────────────┼─────────────────┤
│ layer1.0.conv1.weight        │ (64, 64, 3, 3)   │           0.236 │
├──────────────────────────────┼──────────────────┼─────────────────┤
│ layer1.0.conv2.weight        │ (64, 64, 3, 3)   │           0.110 │
├──────────────────────────────┼──────────────────┼─────────────────┤
│ layer1.1.conv1.weight        │ (64, 64, 3, 3)   │           0.118 │
├──────────────────────────────┼──────────────────┼─────────────────┤
│ layer1.1.conv2.weight        │ (64, 64, 3, 3)   │           0.106 │
├───────────────────

## Fine-tune the Pruned Model

[back to top ⬆️](#Table-of-contents:)

After pruning and BN adaptation, we fine-tune the model for several epochs to recover accuracy lost due to weight removal. After fine-tuning, we strip the pruning masks to permanently apply the sparsity.

In [ ]:
pruning_lr = init_lr / 10
optimizer = torch.optim.Adam(pruned_model.parameters(), lr=pruning_lr)

print("Fine-tuning pruned model...")
for epoch in range(5):
    train(train_loader, pruned_model, criterion, optimizer, epoch=epoch)

acc1_pruned = validate(val_loader, pruned_model, criterion)
print(f"Accuracy of fine-tuned pruned model: {acc1_pruned:.3f}")

# Strip pruning masks to permanently apply sparsity
nncf.strip(pruned_model, strip_format=nncf.StripFormat.IN_PLACE)
print("Pruning masks stripped. Model weights are now permanently sparse.")

## Quantize the Pruned Model

[back to top ⬆️](#Table-of-contents:)

Next, we quantize the pruned model to INT8 using `nncf.quantize()`. This applies post-training quantization, inserting fake-quantize operations into the model. The quantized model can then be fine-tuned to recover accuracy (Quantization-Aware Training).

Evaluate the quantized sparse model on the validation set after initialization.

In [11]:
def transform_fn(data_item):
    return data_item[0].to(device)

calibration_dataset = nncf.Dataset(train_loader, transform_fn)

compressed_model = nncf.quantize(pruned_model, calibration_dataset)

acc1 = validate(val_loader, compressed_model, criterion)
print(f"Accuracy of initialized sparse INT8 model: {acc1:.3f}")

/home/maleksandr/test_notebooks/nncf-tests/openvino_notebooks/venv12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Output()

INFO:nncf:Compiling and loading torch extension: quantized_functions_cpu...
INFO:nncf:Finished loading torch extension: quantized_functions_cpu


Output()

/home/maleksandr/test_notebooks/nncf-tests/openvino_notebooks/venv12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Test: [ 0/79]	Time 0.670 (0.670)	Loss 6.752 (6.752)	Acc@1 2.34 (2.34)	Acc@5 12.50 (12.50)
Test: [10/79]	Time 0.160 (0.177)	Loss 6.697 (6.507)	Acc@1 0.00 (0.71)	Acc@5 7.03 (3.62)
Test: [20/79]	Time 0.126 (0.155)	Loss 7.654 (6.563)	Acc@1 2.34 (1.15)	Acc@5 7.81 (4.50)
Test: [30/79]	Time 0.125 (0.146)	Loss 6.476 (6.693)	Acc@1 0.00 (0.83)	Acc@5 2.34 (3.35)
Test: [40/79]	Time 0.127 (0.142)	Loss 6.153 (6.614)	Acc@1 0.00 (0.74)	Acc@5 1.56 (3.39)
Test: [50/79]	Time 0.129 (0.139)	Loss 5.985 (6.572)	Acc@1 0.00 (0.69)	Acc@5 0.78 (3.23)
Test: [60/79]	Time 0.125 (0.137)	Loss 8.074 (6.572)	Acc@1 0.00 (0.67)	Acc@5 0.00 (3.00)
Test: [70/79]	Time 0.124 (0.136)	Loss 6.824 (6.653)	Acc@1 0.78 (0.61)	Acc@5 2.34 (2.66)
 * Acc@1 0.570 Acc@5 2.640
Accuracy of initialized sparse INT8 model: 0.570


## Fine-tune the Quantized Model (QAT)
[back to top ⬆️](#Table-of-contents:)

At this step, Quantization-Aware Training (QAT) is applied to further improve accuracy of the quantized sparse model. A few epochs of fine-tuning with a small learning rate are usually sufficient.

In [ ]:
compression_lr = init_lr / 10
optimizer = torch.optim.Adam(compressed_model.parameters(), lr=compression_lr)
nr_epochs = 5
# Train for several epochs with NNCF.
print("Training (QAT)")
for epoch in range(nr_epochs):
    train(train_loader, compressed_model, criterion, optimizer, epoch=epoch)

# Evaluate on validation set after Quantization-Aware Training (QAT case).
print("Validating")
acc1_int8_sparse = validate(val_loader, compressed_model, criterion)

print(f"Accuracy of tuned INT8 sparse model: {acc1_int8_sparse:.3f}")
print(f"Accuracy of initialized INT8 sparse model was: {acc1:.3f}")
print(f"QAT improvement: {acc1_int8_sparse - acc1:.3f}")

Training
Epoch:[0][  0/782]	Time 0.829 (0.829)	Loss 5.581 (5.581)	Acc@1 0.00 (0.00)	Acc@5 3.91 (3.91)
Epoch:[0][ 50/782]	Time 0.288 (0.300)	Loss 5.677 (5.657)	Acc@1 0.00 (0.66)	Acc@5 2.34 (2.83)
Epoch:[0][100/782]	Time 0.286 (0.294)	Loss 5.521 (5.625)	Acc@1 0.78 (0.67)	Acc@5 4.69 (2.93)
Epoch:[0][150/782]	Time 0.287 (0.293)	Loss 5.498 (5.586)	Acc@1 0.00 (0.75)	Acc@5 3.91 (3.27)
Epoch:[0][200/782]	Time 0.287 (0.291)	Loss 5.452 (5.554)	Acc@1 0.78 (0.79)	Acc@5 3.12 (3.56)
Epoch:[0][250/782]	Time 0.284 (0.291)	Loss 5.321 (5.519)	Acc@1 3.91 (0.92)	Acc@5 7.03 (3.87)
Epoch:[0][300/782]	Time 0.285 (0.290)	Loss 5.215 (5.487)	Acc@1 1.56 (1.05)	Acc@5 6.25 (4.27)
Epoch:[0][350/782]	Time 0.286 (0.290)	Loss 5.164 (5.451)	Acc@1 1.56 (1.14)	Acc@5 10.16 (4.67)
Epoch:[0][400/782]	Time 0.286 (0.289)	Loss 5.125 (5.420)	Acc@1 3.12 (1.27)	Acc@5 8.59 (5.12)
Epoch:[0][450/782]	Time 0.286 (0.289)	Loss 5.071 (5.388)	Acc@1 3.91 (1.44)	Acc@5 14.84 (5.65)
Epoch:[0][500/782]	Time 0.286 (0.289)	Loss 4.946 (5.358)	Ac

## Export INT8 Sparse Model to OpenVINO IR
[back to top ⬆️](#Table-of-contents:)

In [13]:
warnings.filterwarnings("ignore", category=TracerWarning)
warnings.filterwarnings("ignore", category=UserWarning)
# Export INT8 model to OpenVINO™ IR
ov_model = ov.convert_model(compressed_model, example_input=dummy_input, input=[1, 3, image_size, image_size])
ov.save_model(ov_model, int8_sparse_ir_path)
print(f"INT8 sparse model exported to {int8_sparse_ir_path}.")

INT8 sparse model exported to model/resnet18_int8_sparse.xml.


## Benchmark Model Performance by Computing Inference Time
[back to top ⬆️](#Table-of-contents:)

Finally, measure the inference performance of the `FP32` and `INT8` models, using [Benchmark Tool](https://docs.openvino.ai/2024/learn-openvino/openvino-samples/benchmark-tool.html) - inference performance measurement tool in OpenVINO. By default, Benchmark Tool runs inference for 60 seconds in asynchronous mode on CPU. It returns inference speed as latency (milliseconds per image) and throughput (frames per second) values.

> **NOTE**: This notebook runs `benchmark_app` for 15 seconds to give a quick indication of performance. For more accurate performance, it is recommended to run `benchmark_app` in a terminal/command prompt after closing other applications. Run `benchmark_app -m model.xml -d CPU` to benchmark async inference on CPU for one minute. Change CPU to GPU to benchmark on GPU. Run `benchmark_app --help` to see an overview of all command-line options.

In [14]:
# Initialize OpenVINO runtime
core = ov.Core()
device = device_widget()

device

Dropdown(description='Device:', index=1, options=('CPU', 'AUTO'), value='AUTO')

In [15]:
def parse_benchmark_output(benchmark_output):
    parsed_output = [line for line in benchmark_output if "FPS" in line]
    print(*parsed_output, sep="\n")


print("Benchmark FP32 model (IR)")
benchmark_output = ! benchmark_app -m $fp32_ir_path -d $device.value -api async -t 15
parse_benchmark_output(benchmark_output)

print("Benchmark INT8 sparse model (IR)")
benchmark_output = ! benchmark_app -m $int8_sparse_ir_path -d $device.value -api async -t 15
parse_benchmark_output(benchmark_output)

Benchmark FP32 model (IR)
[ INFO ] Throughput:   3904.80 FPS
Benchmark INT8 sparse model (IR)
[ INFO ] Throughput:   14116.68 FPS


Show Device Information for reference.

In [16]:
import openvino.properties as props


core.get_property(device.value, props.device.full_name)

'AUTO'